# Document Similarity & Bulletin Routing

Given an **incoming** OIR document, decide whether it *updates* an existing
knowledge-base section or is a *new* bulletin.

Pipeline: chunk -> embed (e5) -> vector recall (Qdrant) -> cross-encoder rerank -> LLM route.


In [1]:
from pathlib import Path

# Anchor every path to the project root so the notebook runs from notebook/ or from the repo root.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mock-data").is_dir())
DATA_DIR = PROJECT_ROOT / "mock-data"

# Most specific separators first: spaced sentence endings before bare punctuation,
# so we split on real sentence boundaries rather than inside "NT$1,000." or "Fall 2026!".
SEPARATORS = ["\n\n", "\n", "。", "！", "？", "；",
              ". ", "! ", "? ", "; ", "!", "?", ";", " ", ""]

EMBED_MODEL = "intfloat/multilingual-e5-base"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"   # cross-encoder: scores the pair jointly, far better calibrated than cosine

CHUNK_SIZE_CHARS = 800
CHUNK_OVERLAP_CHARS = 200

COLLECTION = "test_bulletins"

# Retrieval / routing
TOP_K = 5               # candidates recalled per incoming chunk, before reranking
BATCH_SIZE = 32         # encode batch size
UPDATE_THRESHOLD = 0.5  # reranker probability above which we treat a match as "same topic"

print("Project root:", PROJECT_ROOT)


Project root: C:\Users\David Gunawan Wisno\Documents\project\final-project\oir-hub-ai


In [2]:
import os
import torch
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(PROJECT_ROOT / ".env")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


C:\Users\David Gunawan Wisno\Documents\project\final-project\oir-hub-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda NVIDIA GeForce RTX 3070 Laptop GPU


In [3]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

# fp16 on GPU: roughly 2x faster encoding, no measurable retrieval quality loss.
model_kwargs = {"torch_dtype": torch.float16} if DEVICE == "cuda" else {}
model = SentenceTransformer(
    EMBED_MODEL,
    device=DEVICE,
    token=os.getenv("HUGGINGFACE_TOKEN"),
    model_kwargs=model_kwargs,
)

VECTOR_SIZE = model.get_embedding_dimension()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_CHARS,
    chunk_overlap=CHUNK_OVERLAP_CHARS,
    separators=SEPARATORS,
    keep_separator="end",  # sentence-ending punctuation stays attached
)

print("Model loaded:", EMBED_MODEL)
print("Vector size:", VECTOR_SIZE)
print("Qdrant collections:", [c.name for c in client.get_collections().collections])


C:\Users\David Gunawan Wisno\AppData\Local\Temp\ipykernel_10584\2668594960.py:1: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   3%|▎         | 5/199 [00:01<00:39,  4.95it/s]

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 194.57it/s]

Model loaded: intfloat/multilingual-e5-base
Vector size: 768
Qdrant collections: ['oir_kb_e5_base', 'test_bulletins', 'test_collection_e5_base']


In [4]:
from qdrant_client.models import Distance, VectorParams, PayloadSchemaType

client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)

# Indexed so filtered lookups ("everything from this file") stay fast as the KB grows.
client.create_payload_index(
    collection_name=COLLECTION,
    field_name="source_file",
    field_schema=PayloadSchemaType.KEYWORD,
)

print("Fresh collection:", COLLECTION)


Fresh collection: test_bulletins


#### Add Prefix

e5 models are asymmetric: stored text needs `passage: `, search text needs `query: `.
Getting this wrong silently costs several points of accuracy.


In [5]:
def embed_passages(texts, show_progress=False):
    return model.encode(
        [f"passage: {t}" for t in texts],
        normalize_embeddings=True,
        batch_size=BATCH_SIZE,
        show_progress_bar=show_progress,
    )

def embed_queries(texts, show_progress=False):
    return model.encode(
        [f"query: {t}" for t in texts],
        normalize_embeddings=True,
        batch_size=BATCH_SIZE,
        show_progress_bar=show_progress,
    )


### Ingest the knowledge base


In [6]:
import uuid
from qdrant_client.models import PointStruct

documents_to_upload = [
    DATA_DIR / "tunghai_academic_guide.txt",
    DATA_DIR / "tunghai_life_guide.txt",
]


In [7]:
# Chunk every document first, then embed and upsert in ONE batch instead of a
# separate encode + network round-trip per file.
all_chunks, all_meta = [], []

for path in documents_to_upload:
    chunks = splitter.split_text(path.read_text(encoding="utf-8"))
    all_chunks.extend(chunks)
    all_meta.extend((path.name, i) for i in range(len(chunks)))
    print(f"{path.name}: {len(chunks)} chunks")

vectors = embed_passages(all_chunks)

points = [
    PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{name}:{idx}")),
        vector=vec.tolist(),
        payload={"source_file": name, "chunk_index": idx, "content": text},
    )
    for text, vec, (name, idx) in zip(all_chunks, vectors, all_meta)
]

client.upsert(collection_name=COLLECTION, points=points, wait=True)
print("Total points:", client.count(COLLECTION).count)


tunghai_academic_guide.txt: 2 chunks
tunghai_life_guide.txt: 2 chunks


Total points: 4


### Analyse the incoming document

The incoming document is **chunked and matched section by section**. Embedding a whole
document into a single vector averages away the specific claim that changed, and silently
truncates anything past the model's 512-token window - which is exactly the signal we
need in order to route.


In [8]:
incoming_path = DATA_DIR / "mock-arc.txt"
new_document_text = incoming_path.read_text(encoding="utf-8")

incoming_chunks = splitter.split_text(new_document_text)
print(f"{incoming_path.name}: {len(incoming_chunks)} chunk(s), {len(new_document_text)} chars")


mock-arc.txt: 1 chunk(s), 635 chars


In [9]:
from qdrant_client.models import QueryRequest

query_vectors = embed_queries(incoming_chunks)

# One round-trip for every incoming chunk, instead of a query_points call per chunk.
batch_results = client.query_batch_points(
    collection_name=COLLECTION,
    requests=[
        QueryRequest(query=v.tolist(), limit=TOP_K, with_payload=True)
        for v in query_vectors
    ],
)

candidates = [
    {"incoming_index": i, "incoming_text": incoming_chunks[i], "hit": hit}
    for i, result in enumerate(batch_results)
    for hit in result.points
]
print(f"Recalled {len(candidates)} candidate(s) across {len(incoming_chunks)} incoming chunk(s)")


Recalled 4 candidate(s) across 1 incoming chunk(s)


#### Rerank

Cosine similarity from a bi-encoder is poorly calibrated: any two documents from the same
domain land around 0.85-0.95, so thresholding on it is close to meaningless. The
cross-encoder reads both texts *together* and returns a usable probability, which spreads
the scores far enough apart to actually threshold on.


In [10]:
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE, max_length=512)

pairs = [(c["incoming_text"], c["hit"].payload["content"]) for c in candidates]
rerank_scores = reranker.predict(
    pairs,
    activation_fn=torch.nn.Sigmoid(),  # bge-reranker emits logits; squash to 0-1
    batch_size=BATCH_SIZE,
)

for c, s in zip(candidates, rerank_scores):
    c["vector_score"] = c["hit"].score
    c["rerank_score"] = float(s)

candidates.sort(key=lambda c: c["rerank_score"], reverse=True)

print(f"{'rerank':>8}  {'cosine':>7}  source")
for c in candidates[:TOP_K]:
    print(f"{c['rerank_score']:>8.4f}  {c['vector_score']:>7.4f}  "
          f"{c['hit'].payload['source_file']}#{c['hit'].payload['chunk_index']}")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4870.05it/s]

  rerank   cosine  source
  0.7158   0.8795  tunghai_academic_guide.txt#0
  0.4814   0.8438  tunghai_life_guide.txt#1
  0.0090   0.7999  tunghai_academic_guide.txt#1
  0.0003   0.8116  tunghai_life_guide.txt#0


In [11]:
best = candidates[0]

print(f"--- Best match for: {incoming_path.name} ---")
print(f"Rerank score : {best['rerank_score']:.4f}   (cosine was {best['vector_score']:.4f})")
print(f"Source file  : {best['hit'].payload['source_file']}")
print(f"Incoming part: {best['incoming_text'][:200]}...")
print(f"Matched      :\n{best['hit'].payload['content']}")
print("-" * 50)

likely_update = best["rerank_score"] >= UPDATE_THRESHOLD
print("\nHeuristic:", "UPDATE candidate" if likely_update else "looks like a NEW bulletin")


--- Best match for: mock-arc.txt ---
Rerank score : 0.7158   (cosine was 0.8795)
Source file  : tunghai_academic_guide.txt
Incoming part: Important Immigration Update for International Students

Starting Fall 2026, Alien Resident Certificate (ARC) applications will no longer be accepted in person at the Taichung City office. All interna...
Matched      :
Tunghai University — Academic & Immigration Guide

1. ALIEN RESIDENT CERTIFICATE (ARC) REGISTRATION
All international students staying in Taiwan for more than six months must apply for an Alien Resident Certificate within 15 days of arrival. Applications are submitted to the National Immigration Agency office in Taichung City. Processing takes 10 working days, and the fee is NT$1,000.

2. COURSE CREDIT TRANSFER POLICY
Exchange and degree-seeking students may apply to transfer credits earned at their home institution. Applications must be submitted to the Office of Academic Affairs within the first four weeks of the semester. A maximum 

### Route with the LLM

The router is handed the **actual** retrieved chunk and its reranker score, so the decision
is grounded in what is really in the knowledge base.


In [12]:
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal, Optional

# Constrain target_file to files that actually exist in the KB, so the grammar makes it
# impossible for the model to invent a filename (it answered "knowledge_base.json" otherwise).
KNOWN_FILES = sorted({p.name for p in documents_to_upload})

class RoutingDecision(BaseModel):
    action: Literal["UPDATE", "NEW_BULLETIN"]
    target_file: Optional[Literal[tuple(KNOWN_FILES)]]
    reasoning: str
    proposed_content: str

clientLlm = OpenAI(base_url="http://localhost:11434/v1", api_key="unused")

resp = clientLlm.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": (
            "You route incoming OIR documents. Compare the new document against the closest "
            "existing knowledge-base chunk. Choose UPDATE only if the new document supersedes "
            "or contradicts that chunk on the same topic; otherwise choose NEW_BULLETIN. "
            "target_file must be copied verbatim from the source given below, never invented. "
            "For UPDATE, proposed_content must be the full corrected chunk text. "
            "Keep reasoning to one or two sentences about the policy change itself. "
            "Output JSON only, matching the schema exactly."
        )},
        {"role": "user", "content": (
            f"New document ({incoming_path.name}):\n{new_document_text}\n\n"
            f"Closest existing chunk (source: {best['hit'].payload['source_file']}, "
            f"relevance {best['rerank_score']:.2f}):\n{best['hit'].payload['content']}"
        )},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "routing_decision",
            "schema": RoutingDecision.model_json_schema(),
            "strict": True,
        },
    },
    temperature=0,
)

decision = RoutingDecision.model_validate_json(resp.choices[0].message.content)

print("ACTION     :", decision.action)
print("TARGET FILE:", decision.target_file)
print("REASONING  :", decision.reasoning)
print("PROPOSED   :", decision.proposed_content)


ACTION     : UPDATE
TARGET FILE: tunghai_academic_guide.txt
REASONING  : The new document supersedes the existing chunk by updating submission method to online (starting Fall 2026) and adding penalty details for late applications, which directly contradict the previous in-person requirement and penalty information.
PROPOSED   : Tunghai University — Academic & Immigration Guide

1. ALIEN RESIDENT CERTIFICATE (ARC) REGISTRATION
All international students staying in Taiwan for more than six months must apply for an Alien Resident Certificate within 15 days of arrival. Starting Fall 2026, applications are now submitted online via the National Immigration Agency web portal. Processing takes 10 working days, and the fee is NT$1,000. The penalty for late applications has increased: fines for applying after the 15-day arrival window now range from NT$3,000 to NT$15,000.

2. COURSE CREDIT TRANSFER POLICY
Exchange and degree-seeking students may apply to transfer credits earned at their home ins